In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = './gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
from datasets import load_dataset
dataset_name = './sst2'
ds = load_dataset(dataset_name)
ds_train, ds_val = ds['train'], ds['validation']

In [ ]:
ds

In [ ]:
ds_train

In [ ]:
ds_train[6]

In [ ]:
ds_train[:10]

In [ ]:
def tokenize(batch):
    return tokenizer(batch['sentence'])

map_kwargs = {
    'batched': True,
    'batch_size': 512,
    'remove_columns': ['idx', 'sentence', 'label']
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

In [ ]:
tokenized_dataset_train[0]

In [ ]:
tokenized_dataset_train[5:10]

In [ ]:
for i, seq in enumerate(tokenized_dataset_train[5:10]['input_ids']):
    print(f'{i+1}: {tokenizer.decode(seq)}')

In [ ]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))

In [ ]:
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 5)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 5)

In [ ]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))

In [ ]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [ ]:
tokenized_dataset_train[0]

In [ ]:
tokenized_dataset_train[:5]

In [ ]:
tokenizer.pad_token

In [ ]:
tokenizer.eos_token

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False) # labels

dataloader_params = {
    'batch_size': 16,
    'collate_fn': data_collator
}

train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [ ]:
len(train_dataloader)

In [ ]:
3088 * 16

In [ ]:
batch = next(iter(train_dataloader))

In [ ]:
batch.keys()

In [ ]:
batch['input_ids'].shape

In [ ]:
batch['input_ids'][0]

In [ ]:
batch['labels'][0]

In [ ]:
batch['attention_mask'][0]

In [ ]:
import torch
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 1

In [ ]:
def validate(epoch):
    model.eval()
    total_loss = 0.0
    for i, batch in enumerate(val_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss # 损失
            total_loss += loss.item()
    print(f'val_loss at {epoch} epoch:', total_loss / len(val_dataloader))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
validate(0)
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss
        print(f'Loss: {loss.item()}')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    validate(epoch+1)

In [ ]:
model.save_pretrained('./gpt2-sft')
tokenizer.save_pretrained('./gpt2-sft')

In [ ]:
from transformers import pipeline, set_seed
from pprint import pprint
g = pipeline('text-generation', model='./gpt2-sft')
set_seed(1337)
pprint(g("Hi, this is all terribly", max_length=30, num_return_sequences=1))